# alsrory.AI · محرّك التوليد السحابي المجاني

دفتر واحد يحوّل جلسة **Google Colab** أو **Kaggle** إلى محرّك فيديو يعمل مع تطبيق alsrory.AI على جهازك.

**ما يفعله:**
1. يثبّت ComfyUI.
2. ينزّل نموذج **Wan 2.1 T2V 1.3B** (الأخف، مناسب لـ T4 16GB).
3. يشغّل وسيطاً بحماية **رمز سري** لمنع أي شخص يملك الرابط من استخدام المحرّك.
4. يفتح نفقاً عاماً عبر Cloudflare وينتج رابطاً جاهزاً للصق في التطبيق.

**التشغيل:**
- Colab: `Runtime → Change runtime type → T4 GPU`، ثم شغّل الخلايا بالترتيب.
- Kaggle: `Settings → Accelerator → GPU T4 x2`، ثم شغّل الخلايا بالترتيب.

> الجلسة تنتهي عند إغلاق المتصفح أو بعد ساعات الاستخدام المجانية. بعد إعادة التشغيل، أعد تنفيذ الخلايا وألصق الرابط الجديد.

## 1) التحقق من العتاد

In [ ]:
!nvidia-smi
import torch, platform, sys

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda, "| متاح:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU    : {name} ({vram:.1f} GB)")
    assert vram >= 14, "تحتاج ذاكرة 14GB+ — أعد تشغيل الجلسة مع T4/P100."
else:
    raise SystemExit("لا يوجد GPU. فعّل المسرّع من إعدادات بيئة التشغيل.")

## 2) تثبيت ComfyUI

يُستخدم المستودع الرسمي. التثبيت يستغرق 3–6 دقائق في المرة الأولى.

In [ ]:
import os, pathlib, subprocess

COMFY_DIR = pathlib.Path("/content/ComfyUI" if pathlib.Path("/content").exists() else "/kaggle/working/ComfyUI")

if not COMFY_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)],
        check=True,
    )
else:
    print("ComfyUI موجود مسبقاً:", COMFY_DIR)

os.chdir(COMFY_DIR)
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[hf_transfer]" httpx fastapi uvicorn

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("تم التثبيت ✓")

## 3) تنزيل النماذج

| الملف | الحجم | المكان |
|---|---|---|
| `wan2.1_t2v_1.3B_fp16.safetensors` | ~2.8 GB | `models/diffusion_models` |
| `umt5_xxl_fp8_e4m3fn_scaled.safetensors` | ~6.7 GB | `models/text_encoders` |
| `wan_2.1_vae.safetensors` | ~250 MB | `models/vae` |

> على Colab يمكن تركيب Google Drive وتحويل `MODELS_ROOT` إليه لتفادي التنزيل في كل جلسة.

In [ ]:
import pathlib, shutil
from huggingface_hub import hf_hub_download

MODELS_ROOT = COMFY_DIR / "models"  # غيّرها إلى Drive إن أردت التخزين الدائم
REPO = "Comfy-Org/Wan_2.1_ComfyUI_repackaged"

FILES = [
    ("split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors", "diffusion_models"),
    ("split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders"),
    ("split_files/vae/wan_2.1_vae.safetensors", "vae"),
]

for remote, folder in FILES:
    target = MODELS_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    dest = target / pathlib.Path(remote).name
    if dest.exists():
        print("موجود مسبقاً:", dest.name)
        continue
    print(f"→ {remote}")
    cached = hf_hub_download(
        repo_id=REPO,
        filename=remote,
        local_dir=MODELS_ROOT / "_downloads",
    )
    shutil.move(str(cached), str(dest))

print("اكتمل تنزيل النماذج ✓")
!du -sh {MODELS_ROOT}

## 4) تشغيل ComfyUI في الخلفية

يبدأ على المنفذ 8188 داخلياً. إن واجهت نفاد ذاكرة (OOM) أضف `"--lowvram"` إلى قائمة الوسائط.

In [ ]:
import time, urllib.request, subprocess, os, sys

LOG = "/content/comfyui.log" if "/content" in str(COMFY_DIR) else "/kaggle/working/comfyui.log"
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd=str(COMFY_DIR),
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
)
print("PID:", proc.pid, "| السجل:", LOG)

ready = False
for i in range(180):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(2)

assert ready, "لم يبدأ ComfyUI — راجع السجل: " + LOG
print("ComfyUI جاهز ✓")

## 5) وسيط الحماية بالرمز السري

لا نكشف ComfyUI مباشرة. هذا الوسيط يرفض كل طلب لا يحمل الترويسة
`X-Alsrory-Token` بالقيمة الصحيحة، ثم يمرّر الطلب إلى `127.0.0.1:8188`.

> المحرّك في alsrory.AI يرسل هذه الترويسة تلقائياً عندما تضع الرمز في الإعدادات.

In [ ]:
import os, secrets, textwrap, pathlib

WORK = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path("/kaggle/working")
TOKEN = os.environ.get("ALSROY_TOKEN") or secrets.token_urlsafe(18)
os.environ["ALSROY_TOKEN"] = TOKEN

proxy_code = textwrap.dedent('''
    import os
    import httpx
    import uvicorn
    from fastapi import FastAPI, HTTPException, Request
    from fastapi.responses import StreamingResponse
    from starlette.background import BackgroundTask

    UPSTREAM = os.environ.get("COMFY_UPSTREAM", "http://127.0.0.1:8188")
    TOKEN = os.environ["ALSROY_TOKEN"]
    HOP = {
        "host", "content-length", "connection", "accept-encoding",
        "transfer-encoding", "keep-alive", "proxy-authenticate",
        "proxy-authorization", "te", "trailers", "upgrade",
    }

    app = FastAPI(title="alsrory engine proxy")

    @app.get("/_alsrory/health")
    def health():
        return {"ok": True, "service": "alsrory-comfy-proxy"}

    @app.api_route(
        "/{path:path}",
        methods=["GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS", "HEAD"],
    )
    async def proxy(path: str, request: Request):
        if request.headers.get("x-alsrory-token") != TOKEN:
            raise HTTPException(status_code=401, detail="رمز غير صالح")
        client = httpx.AsyncClient(timeout=None)
        try:
            body = await request.body()
            headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP}
            upstream = await client.send(
                client.build_request(
                    request.method,
                    f"{UPSTREAM}/{path}",
                    headers=headers,
                    params=request.query_params,
                    content=body,
                ),
                stream=True,
            )
        except Exception:
            await client.aclose()
            raise HTTPException(status_code=502, detail="تعذّر الوصول إلى ComfyUI")

        async def cleanup():
            await upstream.aclose()
            await client.aclose()

        out_headers = {k: v for k, v in upstream.headers.items() if k.lower() not in HOP}
        return StreamingResponse(
            upstream.aiter_raw(),
            status_code=upstream.status_code,
            headers=out_headers,
            background=BackgroundTask(cleanup),
        )

    if __name__ == "__main__":
        uvicorn.run(app, host="127.0.0.1", port=8190, log_level="warning")
''')

proxy_path = WORK / "alsrory_proxy.py"
proxy_path.write_text(proxy_code, encoding="utf-8")
print("كُتب الملف ✓:", proxy_path)

import subprocess, sys, time, urllib.request
proxy_log = WORK / "proxy.log"
proxy_proc = subprocess.Popen(
    [sys.executable, str(proxy_path)],
    stdout=open(proxy_log, "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(4)

ok = False
for _ in range(20):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8190/_alsrory/health", timeout=3) as r:
            ok = r.status == 200
            if ok:
                break
    except Exception:
        time.sleep(1)
assert ok, f"لم يبدأ الوسيط — راجع {proxy_log}"

print("الوسيط يعمل على 8190 ✓")
print("\nALSROY_TOKEN =", TOKEN)

## 6) فتح النفق العام

ينتج رابطاً مؤقتاً مثل `https://xxxx.trycloudflare.com`. **لا تشاركه** —
من يملكه ويحصل على الرمز يستطيع استخدام المحرّك.

In [ ]:
import re, subprocess, time, pathlib, os, urllib.request

BIN = "/usr/local/bin/cloudflared"
if not pathlib.Path(BIN).exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        BIN,
    )
    os.chmod(BIN, 0o755)

log_path = WORK / "cloudflared.log"
tunnel = subprocess.Popen(
    [BIN, "tunnel", "--url", "http://127.0.0.1:8190", "--no-autoupdate"],
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
)

url = None
pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for _ in range(60):
    text = log_path.read_text(errors="ignore") if log_path.exists() else ""
    found = pattern.search(text)
    if found:
        url = found.group(0)
        break
    time.sleep(2)

if not url:
    print(log_path.read_text(errors="ignore")[-3000:])
    raise RuntimeError("لم يظهر رابط النفق — راجع السجل أعلاه.")

print("=" * 62)
print("ALSROY_ENGINE_URL =", url)
print("ALSROY_TOKEN      =", os.environ["ALSROY_TOKEN"])
print("=" * 62)
print("\nانسخ القيمتين إلى: alsrory.AI → الإعدادات → المحرّكات → ComfyUI")

## 7) اختبار سريع (اختياري)

يتأكد أن الوسيط يحمي المحرّك فعلاً: الطلب بدون رمز يجب أن يفشل بـ 401.

In [ ]:
import httpx, os

url = url if "url" in globals() else os.environ.get("ALSROY_ENGINE_URL", "")
token = os.environ["ALSROY_TOKEN"]

r1 = httpx.get(f"{url}/system_stats", timeout=30)
r2 = httpx.get(f"{url}/system_stats", headers={"X-Alsrory-Token": token}, timeout=30)
print("بدون رمز :", r1.status_code, "(المتوقع 401)")
print("مع الرمز  :", r2.status_code, "(المتوقع 200)")
assert r1.status_code == 401 and r2.status_code == 200
print("\nالمحرّك جاهز للاستخدام ✓")

## ملاحظات

- **إبقاء الجلسة:** لا تغلق تبويب المتصفح. الجلسة تنتهي تلقائياً بعد مدة الاستخدام.
- **إعادة التشغيل:** نفّذ الخلايا 4 → 6 مرة أخرى (النماذج تبقى على القرص داخل الجلسة نفسها). الرابط يتغيّر.
- **أنواع توليد أخرى:** ارفع ملفات workflow إضافية إلى `workers/comfy/workflows/` على جهازك؛ لا تحتاج تعديل هذا الدفتر.
- **نموذج أكبر:** استبدل الملف الأول في خلية النماذج بـ `wan2.1_t2v_14B_fp8_e4m3fn.safetensors` إن كان لديك 24GB+.
- **بدون نفق:** على شبكة محلية، يمكنك تشغيل ComfyUI على `0.0.0.0` ووضع IP الجهاز في الإعدادات مباشرة (دون نفق ودون رمز).